In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND CONFIGURATION
# ===================================================

from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F


"""
Ingest the remaining SemiconPlus batch sources into managed Unity Catalog
Bronze tables while preserving source values and file-level lineage.
"""

CATALOG = "semiconplus_portfolio"
BRONZE_SCHEMA = "bronze"

LANDING_ROOT = (
    "/Volumes/semiconplus_portfolio/landing/external_source"
)

CHECKPOINT_ROOT = (
    "/Volumes/semiconplus_portfolio/"
    "monitoring/checkpoints/autoloader"
)

PIPELINE_RUN_ID = str(uuid4())
PIPELINE_START_TIME = datetime.now(timezone.utc)

print(f"Pipeline run ID: {PIPELINE_RUN_ID}")
print(f"Pipeline start UTC: {PIPELINE_START_TIME.isoformat()}")

In [0]:
# ===================================================
# BLOCK 2 — SOURCE CONFIGURATION
# ===================================================

"""
Define each incrementally processed source, its file format, destination
table, checkpoint, and approved initial-load controls.

Reference files are handled separately because each file represents a
different business entity and therefore has a different structure.
"""

incremental_sources = {
    "equipment_events": {
        "source_path": f"{LANDING_ROOT}/equipment_events",
        "format": "csv",
        "reader_options": {
            "header": "true",
            "delimiter": "\t",
            "inferColumnTypes": "false",
        },
        "target_table": f"{CATALOG}.{BRONZE_SCHEMA}.equipment_events",
        "expected_files": 60,
        "expected_rows": 255_640,
    },
    "unit_test_results": {
        "source_path": f"{LANDING_ROOT}/unit_test_results",
        "format": "json",
        "reader_options": {
            "multiLine": "false",
            "inferColumnTypes": "false",
        },
        "target_table": f"{CATALOG}.{BRONZE_SCHEMA}.unit_test_results",
        "expected_files": 60,
        "expected_rows": 181_250,
    },
    "tester_logs": {
        "source_path": f"{LANDING_ROOT}/tester_logs",
        "format": "text",
        "reader_options": {},
        "target_table": f"{CATALOG}.{BRONZE_SCHEMA}.tester_logs_raw",
        "expected_files": 60,
        "expected_rows": 18_125,
    },
}

for source_name, source_config in incremental_sources.items():
    source_config["schema_location"] = (
        f"{CHECKPOINT_ROOT}/{source_name}/schema"
    )
    source_config["checkpoint_location"] = (
        f"{CHECKPOINT_ROOT}/{source_name}/checkpoint"
    )

print("Remaining batch-source configuration created.")

In [0]:
# ===================================================
# BLOCK 3 — SOURCE FILE PREFLIGHT
# ===================================================

"""
Confirm that each incremental source contains the approved number of
non-empty source files before starting ingestion.
"""

source_file_metrics = {}

for source_name, config in incremental_sources.items():
    files = [
        item
        for item in dbutils.fs.ls(config["source_path"])
        if not item.isDir()
    ]

    file_count = len(files)
    total_bytes = sum(item.size for item in files)

    source_file_metrics[source_name] = {
        "file_count": file_count,
        "total_bytes": total_bytes,
    }

    print(
        f"{source_name}: files={file_count}, "
        f"bytes={total_bytes:,}"
    )

    assert file_count == config["expected_files"], (
        f"{source_name}: expected {config['expected_files']} files, "
        f"found {file_count}."
    )
    assert total_bytes > 0, f"{source_name}: source files are empty."

print("Incremental-source preflight validation passed.")

In [0]:
# ===================================================
# BLOCK 4 — AUTO LOADER INGESTION FUNCTION
# ===================================================

"""
Process one configured source with Auto Loader and append newly discovered
files to its managed Bronze Delta table.

Structured sources retain unexpected fields in _rescued_data. Raw text
sources use a fixed value column and do not require schema evolution.
"""


def ingest_with_auto_loader(source_name, config, pipeline_run_id):
    reader = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", config["format"])
        .option(
            "cloudFiles.schemaLocation",
            config["schema_location"],
        )
    )

    # CSV and JSON sources can receive additional or unexpected fields.
    # Auto Loader retains those fields for later quality investigation.
    if config["format"] in {"csv", "json"}:
        reader = (
            reader
            .option(
                "cloudFiles.schemaEvolutionMode",
                "rescue",
            )
            .option(
                "rescuedDataColumn",
                "_rescued_data",
            )
        )

    # Text input has a fixed single-column structure named value.
    # Schema evolution and rescued-data handling are not supported.
    elif config["format"] == "text":
        reader = reader.option(
            "cloudFiles.schemaEvolutionMode",
            "none",
        )

    for option_name, option_value in config[
        "reader_options"
    ].items():
        reader = reader.option(
            option_name,
            option_value,
        )

    source_stream = reader.load(
        config["source_path"]
    )

    # Add a consistent placeholder because text sources cannot create
    # an Auto Loader rescued-data column.
    if config["format"] == "text":
        source_stream = source_stream.withColumn(
            "_rescued_data",
            F.lit(None).cast("string"),
        )

    bronze_stream = source_stream.select(
        "*",
        F.col("_metadata.file_path").alias(
            "_source_file_path"
        ),
        F.col("_metadata.file_name").alias(
            "_source_file_name"
        ),
        F.col(
            "_metadata.file_modification_time"
        ).alias(
            "_source_file_modification_time"
        ),
        F.current_timestamp().alias(
            "_ingested_at_utc"
        ),
        F.lit(pipeline_run_id).alias(
            "_pipeline_run_id"
        ),
    )

    query = (
        bronze_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            config["checkpoint_location"],
        )
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(config["target_table"])
    )

    query.awaitTermination()

    persisted_df = spark.table(
        config["target_table"]
    )

    result = {
        "source_name": source_name,
        "target_table": config["target_table"],
        "row_count": persisted_df.count(),
        "source_file_count": (
            persisted_df
            .select("_source_file_path")
            .distinct()
            .count()
        ),
    }

    print(result)
    return result

In [0]:
# ===================================================
# BLOCK 5 — INGEST EQUIPMENT EVENTS
# ===================================================

"""
Ingest tab-separated equipment-event records while retaining raw source
values for later equipment-state and duration validation in Silver.
"""

equipment_result = ingest_with_auto_loader(
    "equipment_events",
    incremental_sources["equipment_events"],
    PIPELINE_RUN_ID,
)

assert equipment_result["row_count"] == 255_640
assert equipment_result["source_file_count"] == 60

print("Equipment-event Bronze ingestion passed.")

In [0]:
# ===================================================
# BLOCK 6 — INGEST UNIT-TEST RESULTS
# ===================================================

"""
Ingest JSON unit-test results while retaining nested source structures
for controlled flattening and measurement validation in Silver.
"""

unit_test_result = ingest_with_auto_loader(
    "unit_test_results",
    incremental_sources["unit_test_results"],
    PIPELINE_RUN_ID,
)

assert unit_test_result["row_count"] == 181_250
assert unit_test_result["source_file_count"] == 60

print("Unit-test-result Bronze ingestion passed.")

In [0]:
# ===================================================
# BLOCK 7 — INGEST TESTER LOGS
# ===================================================

"""
Ingest tester log lines as raw text records.

Parsing remains a Silver responsibility so malformed lines remain
available for investigation and quarantine instead of being discarded.
"""

tester_log_result = ingest_with_auto_loader(
    "tester_logs",
    incremental_sources["tester_logs"],
    PIPELINE_RUN_ID,
)

assert tester_log_result["row_count"] == 18_125
assert tester_log_result["source_file_count"] == 60

print("Tester-log Bronze ingestion passed.")

In [0]:
# ===================================================
# BLOCK 8 — LOAD REFERENCE TABLES
# ===================================================

"""
Load each reference entity into an independent managed Bronze table.

Overwrite mode maintains a repeatable reference snapshot while retaining
source-file and pipeline-run lineage.
"""

reference_sources = {
    "devices": (
        f"{LANDING_ROOT}/reference/devices.csv"
    ),
    "equipment": (
        f"{LANDING_ROOT}/reference/equipment.csv"
    ),
    "product_groups": (
        f"{LANDING_ROOT}/reference/product_groups.csv"
    ),
    "sites": (
        f"{LANDING_ROOT}/reference/sites.csv"
    ),
}

expected_reference_counts = {
    "devices": 30,
    "equipment": 24,
    "product_groups": 6,
    "sites": 3,
}

reference_results = []

for entity_name, source_path in reference_sources.items():
    reference_source_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .load(source_path)
    )

    reference_df = reference_source_df.select(
        "*",

        # Retain Unity Catalog-compatible file lineage.
        F.col("_metadata.file_path").alias(
            "_source_file_path"
        ),
        F.col("_metadata.file_name").alias(
            "_source_file_name"
        ),
        F.col(
            "_metadata.file_modification_time"
        ).alias(
            "_source_file_modification_time"
        ),

        # Retain the current ingestion execution details.
        F.current_timestamp().alias(
            "_ingested_at_utc"
        ),
        F.lit(PIPELINE_RUN_ID).alias(
            "_pipeline_run_id"
        ),
    )

    row_count = reference_df.count()
    expected_count = expected_reference_counts[
        entity_name
    ]

    assert row_count == expected_count, (
        f"{entity_name}: expected "
        f"{expected_count} records, "
        f"found {row_count}."
    )

    target_table = (
        f"{CATALOG}.{BRONZE_SCHEMA}."
        f"ref_{entity_name}"
    )

    (
        reference_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    # Confirm that the persisted table matches the source control total.
    persisted_count = spark.table(
        target_table
    ).count()

    assert persisted_count == expected_count, (
        f"{target_table}: expected "
        f"{expected_count} persisted records, "
        f"found {persisted_count}."
    )

    reference_results.append(
        (
            entity_name,
            target_table,
            expected_count,
            persisted_count,
            "PASSED",
        )
    )

display(
    spark.createDataFrame(
        reference_results,
        [
            "entity_name",
            "target_table",
            "expected_count",
            "persisted_count",
            "status",
        ],
    ).orderBy("entity_name")
)

print(
    "Reference Bronze tables written and validated."
)


In [0]:
# ===================================================
# BLOCK 9 — LOAD BINARY DOCUMENTS
# ===================================================

"""
Preserve binary maintenance documents with their content and file-level
metadata for later metadata extraction and document-processing examples.

The dataset is small enough to retain the complete binary payload.
"""

BINARY_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.maintenance_documents_binary"
BINARY_SOURCE_PATH = f"{LANDING_ROOT}/binary_documents"

binary_documents_df = (
    spark.read
    .format("binaryFile")
    .load(BINARY_SOURCE_PATH)
    .select(
        F.col("path").alias("source_file_path"),
        F.regexp_extract(F.col("path"), r"([^/]+)$", 1).alias(
            "source_file_name"
        ),
        F.col("modificationTime").alias("source_modification_time"),
        F.col("length").alias("content_length_bytes"),
        F.col("content").alias("binary_content"),
        F.current_timestamp().alias("_ingested_at_utc"),
        F.lit(PIPELINE_RUN_ID).alias("_pipeline_run_id"),
    )
)

assert binary_documents_df.count() == 25
assert binary_documents_df.filter(
    F.col("content_length_bytes") <= 0
).count() == 0

(
    binary_documents_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BINARY_TABLE)
)

print("Binary-document Bronze ingestion passed.")

In [0]:
# ===================================================
# BLOCK 10 — PRESERVE SOURCE MANIFESTS
# ===================================================

"""
Preserve each source manifest as one binary record.

The two files have different JSON contracts, so retaining the original
payload prevents incompatible structures from being forced into one
relational schema. Specific manifest fields can be parsed separately.
"""

MANIFEST_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.source_manifests_raw"
MANIFEST_SOURCE_PATH = f"{LANDING_ROOT}/manifests"

manifest_df = (
    spark.read
    .format("binaryFile")
    .load(MANIFEST_SOURCE_PATH)
    .select(
        F.col("path").alias("source_file_path"),
        F.regexp_extract(F.col("path"), r"([^/]+)$", 1).alias(
            "source_file_name"
        ),
        F.col("modificationTime").alias("source_modification_time"),
        F.col("length").alias("content_length_bytes"),
        F.decode(F.col("content"), "UTF-8").alias("json_payload"),
        F.current_timestamp().alias("_ingested_at_utc"),
        F.lit(PIPELINE_RUN_ID).alias("_pipeline_run_id"),
    )
)

assert manifest_df.count() == 2
assert manifest_df.filter(F.col("json_payload").isNull()).count() == 0

(
    manifest_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MANIFEST_TABLE)
)

print("Source-manifest preservation passed.")

In [0]:
# ===================================================
# BLOCK 11 — WRITE INGESTION AUDIT RECORDS
# ===================================================

"""
Record source-level execution metrics for operational traceability and
future workflow monitoring.
"""

AUDIT_TABLE = f"{CATALOG}.monitoring.ingestion_audit"

audit_rows = [
    (
        PIPELINE_RUN_ID,
        source_name,
        config["source_path"],
        config["target_table"],
        source_file_metrics[source_name]["file_count"],
        result["row_count"],
        "SUCCEEDED",
        PIPELINE_START_TIME,
        datetime.now(timezone.utc),
    )
    for source_name, config, result in [
        (
            "equipment_events",
            incremental_sources["equipment_events"],
            equipment_result,
        ),
        (
            "unit_test_results",
            incremental_sources["unit_test_results"],
            unit_test_result,
        ),
        (
            "tester_logs",
            incremental_sources["tester_logs"],
            tester_log_result,
        ),
    ]
]

audit_df = spark.createDataFrame(
    audit_rows,
    """
    pipeline_run_id STRING,
    source_name STRING,
    source_path STRING,
    target_table STRING,
    source_file_count LONG,
    target_row_count LONG,
    run_status STRING,
    started_at_utc TIMESTAMP,
    completed_at_utc TIMESTAMP
    """,
)

(
    audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)

display(audit_df.orderBy("source_name"))

print("Ingestion-audit records written.")

In [0]:
# ===================================================
# BLOCK 12 — FINAL INGESTION SUMMARY
# ===================================================

"""
Publish the completed initial-load results for the remaining Bronze
batch sources.
"""

summary_rows = [
    ("equipment_events", 60, 255_640),
    ("unit_test_results", 60, 181_250),
    ("tester_logs_raw", 60, 18_125),
    ("ref_devices", 1, 30),
    ("ref_equipment", 1, 24),
    ("ref_product_groups", 1, 6),
    ("ref_sites", 1, 3),
    ("maintenance_documents_binary", 25, 25),
    ("source_manifests_raw", 2, 2),
]

summary_df = spark.createDataFrame(
    summary_rows,
    ["bronze_dataset", "source_file_count", "row_count"],
)

display(summary_df.orderBy("bronze_dataset"))

PIPELINE_END_TIME = datetime.now(timezone.utc)

print("REMAINING BRONZE INGESTION PASSED")
print(f"Pipeline end UTC: {PIPELINE_END_TIME.isoformat()}")
print(
    "Pipeline duration seconds: "
    f"{(PIPELINE_END_TIME - PIPELINE_START_TIME).total_seconds():.2f}"
)